# Phase 4 — Drift quantification (PSI / KS)


In [ ]:
# --- Setup: mount Drive + install driftbench from the public repo ---
from google.colab import drive  # noqa
drive.mount('/content/drive')

# Install the pinned commit for reproducibility (replace USERNAME and the ref).
!pip -q install "git+https://github.com/USERNAME/drift-conference.git@main"

import os
os.environ['DRIFT_DATA_ROOT'] = '/content/drive/MyDrive/drift-conference/data'
os.environ['DRIFT_REPO_ROOT'] = '/content/drift-conference'  # for results/manifests output
import driftbench as db; print('driftbench', db.__version__)


Per-feature PSI + KS between train and test windows, plus label-distribution PSI. Show that drift magnitude lines up with the performance drop.

In [ ]:
import pandas as pd
from driftbench import (config, feature_drift_report, label_psi,
                        chronological_split)
# Within-dataset (Demo A) drift: earliest train window vs latest test window.
ds='CSE-CIC-IDS2018'
X=pd.read_parquet(config.INTERIM_DIR/ds/'features.parquet')
y=pd.read_parquet(config.INTERIM_DIR/ds/'labels.parquet')['label']
meta=pd.read_parquet(config.INTERIM_DIR/ds/'metadata.parquet')
cs=chronological_split(meta)
rep=feature_drift_report(X.iloc[cs['train']], X.iloc[cs['test']], X.columns)
rep.to_csv(config.RESULTS_DIR/f'drift_within_{ds}.csv', index=False)
print('label PSI (train vs test):', round(label_psi(y.iloc[cs['train']], y.iloc[cs['test']]),4))
rep.head(15)


In [ ]:
# Cross-dataset (Demo B) drift on the common core: 2018 vs 2019.
A,B='CSE-CIC-IDS2018','CIC-DDoS2019'
Xa=pd.read_parquet(config.PROCESSED_DIR/A/'features_core.parquet')
Xb=pd.read_parquet(config.PROCESSED_DIR/B/'features_core.parquet')
rep2=feature_drift_report(Xa, Xb, Xa.columns)
rep2.to_csv(config.RESULTS_DIR/'drift_cross_2018_2019.csv', index=False)
rep2.head(15)
